In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [5]:
df_transactions = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/bronze/raw/transactions_raw.csv")
)

print("Total rows:", df_transactions.count())

df_transactions.printSchema()

display(df_transactions.limit(10))

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 7, Finished, Available, Finished, False)

Total rows: 100250
root
 |-- TransactionID: string (nullable = true)
 |-- AccountID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- TransactionType: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- BranchID: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- TransactionStatus: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 494eb186-8397-468e-94b9-326fc08e846a)

In [3]:
files = mssparkutils.fs.ls("Files/bronze/raw")

for file in files:
    print(file.name)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 5, Finished, Available, Finished, False)

accounts[1].csv
branches[1].csv
customers[1].csv
date[1].csv
fraud_transactions[1].csv
loans[1].csv
payments[1].csv
transactions_raw[1].csv


In [6]:
duplicate_count = (
    df_transactions
    .groupBy("TransactionID")
    .count()
    .filter("count > 1")
    .count()
)

print("Duplicate Transaction IDs:", duplicate_count)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 8, Finished, Available, Finished, False)

Duplicate Transaction IDs: 250


In [7]:
duplicate_transactions = (
    df_transactions
    .groupBy("TransactionID")
    .count()
    .filter("count > 1")
    .orderBy("count", ascending=False)
)

display(duplicate_transactions)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 87fdd9d8-d034-4d30-a27c-286bc46c5d39)

In [8]:
duplicate_rows = (
    duplicate_transactions
    .selectExpr("sum(count - 1) as duplicate_rows")
    .collect()[0]["duplicate_rows"]
)

print("Extra duplicate rows:", duplicate_rows)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 10, Finished, Available, Finished, False)

Extra duplicate rows: 250


In [9]:
from pyspark.sql.functions import col, sum, when

null_counts = df_transactions.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_transactions.columns
])

display(null_counts)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a5375c96-91d1-4ef4-9a0c-a9b0428ec1f0)

In [10]:
df_transactions.filter(
    col("Amount").isNull() | col("PaymentMethod").isNull()
).select(
    "TransactionID",
    "Amount",
    "PaymentMethod"
).show(20, truncate=False)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 12, Finished, Available, Finished, False)

+-------------+--------+-------------+
|TransactionID|Amount  |PaymentMethod|
+-------------+--------+-------------+
|TXN000517    |316.12  |NULL         |
|TXN000824    |6733.67 |NULL         |
|TXN000983    |2778.39 |NULL         |
|TXN001102    |830.31  |NULL         |
|TXN001123    |NULL    |IMPS         |
|TXN001352    |NULL    |Cash         |
|TXN001384    |22887.57|NULL         |
|TXN001453    |NULL    |UPI          |
|TXN001798    |NULL    |IMPS         |
|TXN002662    |5342.11 |NULL         |
|TXN002753    |NULL    |UPI          |
|TXN002803    |3362.76 |NULL         |
|TXN002913    |1474.4  |NULL         |
|TXN003481    |NULL    |Debit Card   |
|TXN003845    |1268.41 |NULL         |
|TXN003922    |1090.84 |NULL         |
|TXN004452    |1766.12 |NULL         |
|TXN004542    |NULL    |NEFT         |
|TXN004590    |NULL    |Cash         |
|TXN005298    |1142.79 |NULL         |
+-------------+--------+-------------+
only showing top 20 rows



In [11]:
df_transactions.filter(
    col("Amount").isNull()
).select(
    "TransactionID",
    "TransactionType",
    "PaymentMethod",
    "TransactionStatus"
).show(20, truncate=False)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 13, Finished, Available, Finished, False)

+-------------+---------------+-------------+-----------------+
|TransactionID|TransactionType|PaymentMethod|TransactionStatus|
+-------------+---------------+-------------+-----------------+
|TXN001123    |Withdrawal     |IMPS         |Failed           |
|TXN001352    |Payment        |Cash         |Completed        |
|TXN001453    |Payment        |UPI          |Completed        |
|TXN001798    |Payment        |IMPS         |Completed        |
|TXN002753    |Withdrawal     |UPI          |Completed        |
|TXN003481    |Payment        |Debit Card   |Completed        |
|TXN004542    |Transfer       |NEFT         |Completed        |
|TXN004590    |Transfer       |Cash         |Completed        |
|TXN005765    |Withdrawal     |UPI          |Completed        |
|TXN006435    |Deposit        |UPI          |Completed        |
|TXN006704    |Payment        |NEFT         |Completed        |
|TXN006971    |Withdrawal     |NEFT         |Completed        |
|TXN008815    |Deposit        |UPI      

In [12]:
from pyspark.sql.functions import col, when

df_silver = df_transactions.withColumn(
    "PaymentMethod",
    when(col("PaymentMethod").isNull(), "Unknown")
    .otherwise(col("PaymentMethod"))
)

display(df_silver.limit(10))

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 91da9681-dafe-4cd0-8f73-a62f55bea145)

In [13]:
df_silver = df_silver.dropDuplicates(["TransactionID"])

print("Rows after removing duplicates:", df_silver.count())

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 15, Finished, Available, Finished, False)

Rows after removing duplicates: 100000


In [14]:
duplicate_count = (
    df_silver
    .groupBy("TransactionID")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate Transaction IDs after cleaning:", duplicate_count)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 16, Finished, Available, Finished, False)

Duplicate Transaction IDs after cleaning: 0


In [15]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.transactions")

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 17, Finished, Available, Finished, False)

In [16]:
display(spark.sql("""
    SELECT *
    FROM silver.transactions
    LIMIT 10
"""))

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4fe415ce-8ffa-4c31-9363-c651798a8a43)

In [17]:
from pyspark.sql.functions import sum, count

gold_transaction_summary = (
    spark.table("silver.transactions")
    .groupBy("TransactionType")
    .agg(
        count("*").alias("TotalTransactions"),
        sum("Amount").alias("TotalAmount")
    )
)

display(gold_transaction_summary)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, edc632ae-6008-44a0-a56b-e1c6ba8a75a6)

In [18]:
gold_transaction_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.transaction_summary")

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 20, Finished, Available, Finished, False)

In [19]:
gold_transaction_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.transaction_summary")

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 21, Finished, Available, Finished, False)

In [20]:
display(spark.sql("""
    SELECT *
    FROM gold.transaction_summary
"""))

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 53ae87ce-18e7-4cc8-a5b8-f9ed08072f0f)

In [21]:
gold_payment_summary = (
    spark.table("silver.transactions")
    .groupBy("PaymentMethod")
    .agg(
        count("*").alias("TotalTransactions"),
        sum("Amount").alias("TotalAmount")
    )
)

display(gold_payment_summary)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 04d037ea-2750-4cde-8b98-568255033a48)

In [22]:
gold_payment_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.payment_summary")

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 24, Finished, Available, Finished, False)

In [23]:
display(spark.sql("""
    SELECT *
    FROM gold.payment_summary
"""))

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b0b990a4-6253-41ae-889f-d40b38842c9f)

In [24]:
gold_status_summary = (
    spark.table("silver.transactions")
    .groupBy("TransactionStatus")
    .agg(
        count("*").alias("TotalTransactions"),
        sum("Amount").alias("TotalAmount")
    )
)

display(gold_status_summary)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f54a5ab1-7c12-4825-95ec-e28ba99e1e3f)

In [25]:
from pyspark.sql.functions import col, initcap

df_silver = df_silver.withColumn(
    "TransactionStatus",
    initcap(col("TransactionStatus"))
)

display(df_silver.select("TransactionStatus").distinct())

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 48256ae6-a228-4279-a249-b86d8dd3e26f)

In [26]:
gold_status_summary = (
    df_silver
    .groupBy("TransactionStatus")
    .agg(
        count("*").alias("TotalTransactions"),
        sum("Amount").alias("TotalAmount")
    )
)

display(gold_status_summary)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d9ec20dd-3b8d-45e0-a896-e98f457a3b90)

In [27]:
gold_status_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.status_summary")

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 29, Finished, Available, Finished, False)

In [28]:
display(spark.sql("""
SELECT *
FROM gold.status_summary
"""))

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d97f5c89-5dd6-401c-96d5-89f8309d6e8a)

In [29]:
gold_branch_summary = (
    spark.table("silver.transactions")
    .groupBy("BranchID")
    .agg(
        count("*").alias("TotalTransactions"),
        sum("Amount").alias("TotalAmount")
    )
)

display(gold_branch_summary)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 23426fc5-b6fe-4181-8c50-3b8282aeeac9)

In [30]:
gold_branch_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.branch_summary")

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 32, Finished, Available, Finished, False)

In [31]:
display(spark.sql("""
SELECT *
FROM gold.branch_summary
LIMIT 10
"""))

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9829978f-5203-41b5-8d35-ba92bdf8c936)

In [32]:
gold_date_summary = (
    spark.table("silver.transactions")
    .groupBy("TransactionDate")
    .agg(
        count("*").alias("TotalTransactions"),
        sum("Amount").alias("TotalAmount")
    )
)

display(gold_date_summary)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 115cf723-315a-4685-861c-6b9a9153697e)

In [33]:
gold_date_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.date_summary")

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 35, Finished, Available, Finished, False)

In [34]:
display(spark.sql("""
    SELECT *
    FROM gold.date_summary
    LIMIT 10
"""))

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 77a5edc8-26fb-40b9-ba1f-a98b8f579a2e)

In [35]:
gold_customer_summary = (
    spark.table("silver.transactions")
    .groupBy("CustomerID")
    .agg(
        count("*").alias("TotalTransactions"),
        sum("Amount").alias("TotalAmount")
    )
)

display(gold_customer_summary)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4de0e5d4-6a6d-4f17-a3a7-a9877c865c51)

In [36]:
gold_customer_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.customer_summary")

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 38, Finished, Available, Finished, False)

In [37]:
display(spark.sql("""
    SELECT *
    FROM gold.customer_summary
    LIMIT 10
"""))

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0fae59ca-4ea6-4da0-9984-05f8325ec732)

In [38]:
gold_account_summary = (
    spark.table("silver.transactions")
    .groupBy("AccountID")
    .agg(
        count("*").alias("TotalTransactions"),
        sum("Amount").alias("TotalAmount")
    )
)

display(gold_account_summary)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 78bee6a6-5d28-408f-b4e8-62039751e6e3)

In [39]:
gold_account_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.account_summary")

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 41, Finished, Available, Finished, False)

In [40]:
display(spark.sql("""
    SELECT *
    FROM gold.account_summary
    LIMIT 10
"""))

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 910a7cb9-47a4-4a62-ba23-9bbec759bdf2)

In [41]:
gold_loan_summary = (
    spark.table("silver.transactions")
    .groupBy("LoanID")
    .agg(
        count("*").alias("TotalTransactions"),
        sum("Amount").alias("TotalAmount")
    )
)

display(gold_loan_summary)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 43, Finished, Available, Finished, False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `LoanID` cannot be resolved. Did you mean one of the following? [`Amount`, `BranchID`, `AccountID`, `Location`, `CustomerID`].;
'Aggregate ['LoanID], ['LoanID, count(1) AS TotalTransactions#11460L, sum(Amount#11434) AS TotalAmount#11462]
+- SubqueryAlias spark_catalog.chimcobldhq2agj1dplmirj7bt36irj1dphmiobcbt0msobcf5q6iorj4l162rjbd5n6enqcc5lmaq3felpma9bjd5m7cpbi.transactions
   +- Relation spark_catalog.chimcobldhq2agj1dplmirj7bt36irj1dphmiobcbt0msobcf5q6iorj4l162rjbd5n6enqcc5lmaq3felpma9bjd5m7cpbi.transactions[TransactionID#11429,AccountID#11430,CustomerID#11431,TransactionDate#11432,TransactionType#11433,Amount#11434,PaymentMethod#11435,BranchID#11436,Location#11437,TransactionStatus#11438] parquet


In [42]:
spark.sql("SHOW TABLES IN silver").show()

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 44, Finished, Available, Finished, False)

+--------------------+------------+-----------+
|           namespace|   tableName|isTemporary|
+--------------------+------------+-----------+
|Banking_Financial...|transactions|      false|
+--------------------+------------+-----------+



In [43]:
spark.table("silver.transactions").printSchema()

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 45, Finished, Available, Finished, False)

root
 |-- TransactionID: string (nullable = true)
 |-- AccountID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- TransactionType: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- BranchID: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- TransactionStatus: string (nullable = true)



In [44]:
gold_location_summary = (
    spark.table("silver.transactions")
    .groupBy("Location")
    .agg(
        count("*").alias("TotalTransactions"),
        sum("Amount").alias("TotalAmount")
    )
)

display(gold_location_summary)

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 46, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 802fa088-9b0c-4da6-bf15-f85786004d9a)

In [45]:
gold_location_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.location_summary")

StatementMeta(, 7fd9d110-f801-4d3b-b03b-ee0b6b057eb7, 47, Finished, Available, Finished, False)

In [1]:
df_transactions_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.transactions")

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 3, Finished, Available, Finished, False)

NameError: name 'df_transactions_clean' is not defined

In [2]:
df_transactions_clean = df_transactions.dropDuplicates(["TransactionID"])

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 4, Finished, Available, Finished, False)

NameError: name 'df_transactions' is not defined

In [3]:
df_transactions = spark.read.csv(
    "Files/bronze/raw/transactions_raw[1].csv",
    header=True,
    inferSchema=True
)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 5, Finished, Available, Finished, False)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: abfss://036c253e-83ac-419f-9b73-170095aa78d5@onelake.dfs.fabric.microsoft.com/d5e007f4-972e-476b-a8f7-66e1b947cffc/Files/bronze/raw/transactions_raw[1].csv.

In [4]:
df_transactions = spark.read.csv(
    "Files/bronze/raw/transactions_raw.csv",
    header=True,
    inferSchema=True
)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 6, Finished, Available, Finished, False)

In [5]:
df_transactions_clean = df_transactions.dropDuplicates(["TransactionID"])

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 7, Finished, Available, Finished, False)

In [6]:
df_transactions_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.transactions")

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 8, Finished, Available, Finished, False)

In [7]:
spark.table("silver.transactions").show(10)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 9, Finished, Available, Finished, False)

+-------------+---------+----------+---------------+---------------+--------+-------------+--------+---------+-----------------+
|TransactionID|AccountID|CustomerID|TransactionDate|TransactionType|  Amount|PaymentMethod|BranchID| Location|TransactionStatus|
+-------------+---------+----------+---------------+---------------+--------+-------------+--------+---------+-----------------+
|    TXN000004| ACC11905| CUST08343|     2026-06-05|        Payment| 2639.68|   Debit Card|   BR018|     Pune|           Failed|
|    TXN000007| ACC05261| CUST05563|     2026-07-10|     Withdrawal| 2070.14|          UPI|   BR043|Mangaluru|        Completed|
|    TXN000009| ACC04858| CUST05887|     2026-01-13|        Payment|  366.25|   Debit Card|   BR025|Hyderabad|        Completed|
|    TXN000011| ACC12739| CUST06756|     2024-03-24|        Deposit|  824.29|          ATM|   BR066|     Pune|        Completed|
|    TXN000015| ACC11629| CUST04864|     2024-11-06|        Payment|  640.57|          UPI|   BR0

In [8]:
print("Silver row count:", spark.table("silver.transactions").count())

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 10, Finished, Available, Finished, False)

Silver row count: 100000


In [9]:
from pyspark.sql.functions import col, sum

df_silver = spark.table("silver.transactions")

df_silver.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_silver.columns
]).show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 11, Finished, Available, Finished, False)

+-------------+---------+----------+---------------+---------------+------+-------------+--------+--------+-----------------+
|TransactionID|AccountID|CustomerID|TransactionDate|TransactionType|Amount|PaymentMethod|BranchID|Location|TransactionStatus|
+-------------+---------+----------+---------------+---------------+------+-------------+--------+--------+-----------------+
|            0|        0|         0|              0|              0|   100|          248|       0|       0|                0|
+-------------+---------+----------+---------------+---------------+------+-------------+--------+--------+-----------------+



In [10]:
df_silver.filter(
    col("Amount").isNull() | col("PaymentMethod").isNull()
).show(20, truncate=False)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 12, Finished, Available, Finished, False)

+-------------+---------+----------+---------------+---------------+-------+-------------+--------+----------+-----------------+
|TransactionID|AccountID|CustomerID|TransactionDate|TransactionType|Amount |PaymentMethod|BranchID|Location  |TransactionStatus|
+-------------+---------+----------+---------------+---------------+-------+-------------+--------+----------+-----------------+
|TXN001102    |ACC04004 |CUST08345 |2024-01-19     |Transfer       |830.31 |NULL         |BR059   |Mangaluru |Completed        |
|TXN001352    |ACC07492 |CUST01263 |2025-03-09     |Payment        |NULL   |Cash         |BR065   |Hyderabad |Completed        |
|TXN002913    |ACC08826 |CUST00175 |2026-07-26     |Withdrawal     |1474.4 |NULL         |BR006   |Delhi     |Completed        |
|TXN005298    |ACC05767 |CUST02122 |2025-01-02     |Transfer       |1142.79|NULL         |BR070   |Delhi     |Completed        |
|TXN005765    |ACC13847 |CUST09176 |2026-03-03     |Withdrawal     |NULL   |UPI          |BR050  

In [11]:
df_silver.filter(
    col("Amount").isNull() & col("PaymentMethod").isNull()
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 13, Finished, Available, Finished, False)

0

In [12]:
df_silver.filter(
    col("Amount").isNull()
).groupBy(
    "TransactionType"
).count().show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 14, Finished, Available, Finished, False)

+---------------+-----+
|TransactionType|count|
+---------------+-----+
|        Deposit|   31|
|       Transfer|   28|
|        Payment|   20|
|     Withdrawal|   21|
+---------------+-----+



In [13]:
df_silver.select(
    "Amount"
).summary(
    "count", "min", "avg", "max"
).show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 15, Finished, Available, Finished, False)

IllegalArgumentException: avg is not a recognised statistic.

In [14]:
from pyspark.sql.functions import count, min, avg, max

df_silver.select(
    count("Amount").alias("count"),
    min("Amount").alias("min"),
    avg("Amount").alias("avg"),
    max("Amount").alias("max")
).show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 16, Finished, Available, Finished, False)

+-----+-----+-----------------+---------+
|count|  min|              avg|      max|
+-----+-----+-----------------+---------+
|99900|26.27|5451.613128728724|335240.64|
+-----+-----+-----------------+---------+



In [15]:
df_silver.filter(
    col("Amount").isNull()
).groupBy(
    "TransactionStatus"
).count().show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 17, Finished, Available, Finished, False)

+-----------------+-----+
|TransactionStatus|count|
+-----------------+-----+
|        Completed|   93|
|           Failed|    4|
|          Pending|    3|
+-----------------+-----+



In [16]:
df_silver.filter(
    col("Amount").isNull()
).select(
    "TransactionID",
    "AccountID",
    "CustomerID",
    "TransactionType",
    "PaymentMethod",
    "TransactionStatus"
).show(20, truncate=False)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 18, Finished, Available, Finished, False)

+-------------+---------+----------+---------------+-------------+-----------------+
|TransactionID|AccountID|CustomerID|TransactionType|PaymentMethod|TransactionStatus|
+-------------+---------+----------+---------------+-------------+-----------------+
|TXN001352    |ACC07492 |CUST01263 |Payment        |Cash         |Completed        |
|TXN005765    |ACC13847 |CUST09176 |Withdrawal     |UPI          |Completed        |
|TXN016615    |ACC01986 |CUST04124 |Deposit        |Credit Card  |Completed        |
|TXN024830    |ACC05499 |CUST00479 |Payment        |Debit Card   |Completed        |
|TXN029380    |ACC05960 |CUST09877 |Withdrawal     |IMPS         |Completed        |
|TXN031819    |ACC10580 |CUST09353 |Transfer       |UPI          |Completed        |
|TXN037601    |ACC14188 |CUST03496 |Withdrawal     |UPI          |Completed        |
|TXN040843    |ACC14175 |CUST07044 |Withdrawal     |UPI          |Completed        |
|TXN061542    |ACC00411 |CUST09874 |Withdrawal     |NEFT         

In [17]:
from pyspark.sql.functions import col, count, sum, when

missing_ids = (
    df_silver
    .filter(col("Amount").isNull())
    .select("TransactionID")
    .distinct()
)

recovery_check = (
    df_transactions
    .join(missing_ids, "TransactionID")
    .groupBy("TransactionID")
    .agg(
        count("*").alias("raw_record_count"),
        sum(
            when(col("Amount").isNotNull(), 1).otherwise(0)
        ).alias("amount_available_count")
    )
    .filter(col("amount_available_count") > 0)
)

recovery_check.show(20, truncate=False)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 19, Finished, Available, Finished, False)

+-------------+----------------+----------------------+
|TransactionID|raw_record_count|amount_available_count|
+-------------+----------------+----------------------+
+-------------+----------------+----------------------+



In [18]:
from pyspark.sql.functions import col, sum, when

df_silver.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_silver.columns
]).show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 20, Finished, Available, Finished, False)

+-------------+---------+----------+---------------+---------------+------+-------------+--------+--------+-----------------+
|TransactionID|AccountID|CustomerID|TransactionDate|TransactionType|Amount|PaymentMethod|BranchID|Location|TransactionStatus|
+-------------+---------+----------+---------------+---------------+------+-------------+--------+--------+-----------------+
|            0|        0|         0|              0|              0|   100|          248|       0|       0|                0|
+-------------+---------+----------+---------------+---------------+------+-------------+--------+--------+-----------------+



In [19]:
df_silver.filter(
    col("PaymentMethod").isNull()
).groupBy(
    "TransactionType"
).count().show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 21, Finished, Available, Finished, False)

+---------------+-----+
|TransactionType|count|
+---------------+-----+
|        Deposit|   63|
|       Transfer|   69|
|        Payment|   56|
|     Withdrawal|   60|
+---------------+-----+



In [20]:
df_transactions.filter(
    col("PaymentMethod").isNull()
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 22, Finished, Available, Finished, False)

250

In [21]:
df_transactions.filter(
    col("PaymentMethod").isNull()
).groupBy(
    "TransactionID"
).count().filter(
    col("count") > 1
).show(truncate=False)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 23, Finished, Available, Finished, False)

+-------------+-----+
|TransactionID|count|
+-------------+-----+
+-------------+-----+



In [22]:
df_silver.groupBy("TransactionID") \
    .count() \
    .filter(col("count") > 1) \
    .count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 24, Finished, Available, Finished, False)

0

In [23]:
df_silver.filter(
    col("Amount") < 0
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 25, Finished, Available, Finished, False)

0

In [24]:
df_silver.groupBy("TransactionType").count().show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 26, Finished, Available, Finished, False)

+---------------+-----+
|TransactionType|count|
+---------------+-----+
|        Deposit|28009|
|       Transfer|27108|
|        Payment|19936|
|     Withdrawal|24947|
+---------------+-----+



In [25]:
df_silver.groupBy("TransactionStatus").count().show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 27, Finished, Available, Finished, False)

+-----------------+-----+
|TransactionStatus|count|
+-----------------+-----+
|        completed|   80|
|        Completed|93815|
|           Failed| 3099|
|          Pending| 3006|
+-----------------+-----+



In [26]:
from pyspark.sql.functions import col, when, upper

df_silver = df_silver.withColumn(
    "TransactionStatus",
    when(upper(col("TransactionStatus")) == "COMPLETED", "Completed")
    .when(upper(col("TransactionStatus")) == "FAILED", "Failed")
    .when(upper(col("TransactionStatus")) == "PENDING", "Pending")
    .otherwise(col("TransactionStatus"))
)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 28, Finished, Available, Finished, False)

In [27]:
df_silver.groupBy("TransactionStatus").count().show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 29, Finished, Available, Finished, False)

+-----------------+-----+
|TransactionStatus|count|
+-----------------+-----+
|        Completed|93895|
|           Failed| 3099|
|          Pending| 3006|
+-----------------+-----+



In [28]:
df_silver.filter(
    ~col("TransactionStatus").isin("Completed", "Failed", "Pending")
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 30, Finished, Available, Finished, False)

0

In [29]:
df_silver.groupBy("PaymentMethod").count().show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 31, Finished, Available, Finished, False)

+-------------+-----+
|PaymentMethod|count|
+-------------+-----+
|         IMPS|12874|
|  Credit Card| 8050|
|          ATM|11849|
|         NULL|  248|
|         NEFT|12014|
|         Cash| 8051|
|   Debit Card|12143|
|          UPI|34771|
+-------------+-----+



In [30]:
df_silver.filter(
    col("PaymentMethod").isNotNull() &
    ~col("PaymentMethod").isin(
        "UPI",
        "IMPS",
        "NEFT",
        "ATM",
        "Debit Card",
        "Credit Card",
        "Cash"
    )
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 32, Finished, Available, Finished, False)

0

In [31]:
df_silver.filter(
    col("PaymentMethod").isNull()
).select(
    "TransactionID",
    "AccountID",
    "CustomerID",
    "TransactionType",
    "Amount",
    "TransactionStatus"
).show(20, truncate=False)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 33, Finished, Available, Finished, False)

+-------------+---------+----------+---------------+--------+-----------------+
|TransactionID|AccountID|CustomerID|TransactionType|Amount  |TransactionStatus|
+-------------+---------+----------+---------------+--------+-----------------+
|TXN001102    |ACC04004 |CUST08345 |Transfer       |830.31  |Completed        |
|TXN002913    |ACC08826 |CUST00175 |Withdrawal     |1474.4  |Completed        |
|TXN005298    |ACC05767 |CUST02122 |Transfer       |1142.79 |Completed        |
|TXN015136    |ACC01941 |CUST03808 |Payment        |3627.34 |Completed        |
|TXN019875    |ACC01616 |CUST09912 |Deposit        |1284.32 |Completed        |
|TXN020396    |ACC09212 |CUST05109 |Withdrawal     |4540.97 |Completed        |
|TXN021001    |ACC12968 |CUST00319 |Transfer       |2273.24 |Completed        |
|TXN021880    |ACC05838 |CUST08397 |Deposit        |4741.57 |Completed        |
|TXN022344    |ACC04528 |CUST04035 |Transfer       |1038.45 |Failed           |
|TXN022555    |ACC04450 |CUST06968 |Paym

In [32]:
df_transactions.filter(
    col("PaymentMethod").isNull()
).select(
    "TransactionID",
    "PaymentMethod"
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 34, Finished, Available, Finished, False)

250

In [33]:
df_transactions.filter(
    col("PaymentMethod").isNull()
).groupBy(
    "TransactionType"
).count().show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 35, Finished, Available, Finished, False)

+---------------+-----+
|TransactionType|count|
+---------------+-----+
|        Deposit|   63|
|       Transfer|   69|
|        Payment|   58|
|     Withdrawal|   60|
+---------------+-----+



In [34]:
df_silver.filter(
    col("TransactionDate").isNull()
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 36, Finished, Available, Finished, False)

0

In [35]:
df_silver.select(
    sum(col("AccountID").isNull().cast("int")).alias("AccountID_nulls"),
    sum(col("CustomerID").isNull().cast("int")).alias("CustomerID_nulls")
).show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 37, Finished, Available, Finished, False)

+---------------+----------------+
|AccountID_nulls|CustomerID_nulls|
+---------------+----------------+
|              0|               0|
+---------------+----------------+



In [36]:
df_silver.filter(
    col("TransactionType").isNull()
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 38, Finished, Available, Finished, False)

0

In [37]:
df_silver.groupBy("TransactionType").count().show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 39, Finished, Available, Finished, False)

+---------------+-----+
|TransactionType|count|
+---------------+-----+
|        Deposit|28009|
|       Transfer|27108|
|        Payment|19936|
|     Withdrawal|24947|
+---------------+-----+



In [38]:
df_silver.filter(
    col("PaymentMethod").isNotNull()
).groupBy(
    "PaymentMethod"
).count().show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 40, Finished, Available, Finished, False)

+-------------+-----+
|PaymentMethod|count|
+-------------+-----+
|         IMPS|12874|
|  Credit Card| 8050|
|          ATM|11849|
|         NEFT|12014|
|         Cash| 8051|
|   Debit Card|12143|
|          UPI|34771|
+-------------+-----+



In [39]:
df_silver.select(
    sum(col("BranchID").isNull().cast("int")).alias("BranchID_nulls"),
    sum(col("Location").isNull().cast("int")).alias("Location_nulls")
).show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 41, Finished, Available, Finished, False)

+--------------+--------------+
|BranchID_nulls|Location_nulls|
+--------------+--------------+
|             0|             0|
+--------------+--------------+



In [40]:
df_silver.printSchema()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 42, Finished, Available, Finished, False)

root
 |-- TransactionID: string (nullable = true)
 |-- AccountID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- TransactionType: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- BranchID: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- TransactionStatus: string (nullable = true)



In [41]:
df_silver.groupBy("Location").count().show(50, truncate=False)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 43, Finished, Available, Finished, False)

+----------+-----+
|Location  |count|
+----------+-----+
|Vadodara  |12220|
|Salem     |12531|
|Pune      |12889|
|Delhi     |12058|
|Siliguri  |11936|
|Hyderabad |13464|
|Mangaluru |12589|
|Vijayawada|12313|
+----------+-----+



In [42]:
df_silver.groupBy("PaymentMethod").count().show(50, truncate=False)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 44, Finished, Available, Finished, False)

+-------------+-----+
|PaymentMethod|count|
+-------------+-----+
|IMPS         |12874|
|Credit Card  |8050 |
|ATM          |11849|
|NULL         |248  |
|NEFT         |12014|
|Cash         |8051 |
|Debit Card   |12143|
|UPI          |34771|
+-------------+-----+



In [43]:
df_silver.groupBy("PaymentMethod").count().show(50, truncate=False)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 45, Finished, Available, Finished, False)

+-------------+-----+
|PaymentMethod|count|
+-------------+-----+
|IMPS         |12874|
|Credit Card  |8050 |
|ATM          |11849|
|NULL         |248  |
|NEFT         |12014|
|Cash         |8051 |
|Debit Card   |12143|
|UPI          |34771|
+-------------+-----+



In [44]:
df_silver.filter(
    col("PaymentMethod").isNull()
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 46, Finished, Available, Finished, False)

248

In [45]:
df_silver.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df_silver.columns
    ]
).show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 47, Finished, Available, Finished, False)

+-------------+---------+----------+---------------+---------------+------+-------------+--------+--------+-----------------+
|TransactionID|AccountID|CustomerID|TransactionDate|TransactionType|Amount|PaymentMethod|BranchID|Location|TransactionStatus|
+-------------+---------+----------+---------------+---------------+------+-------------+--------+--------+-----------------+
|            0|        0|         0|              0|              0|   100|          248|       0|       0|                0|
+-------------+---------+----------+---------------+---------------+------+-------------+--------+--------+-----------------+



In [46]:
df_silver.filter(
    col("Amount").isNull()
).select(
    "TransactionID",
    "AccountID",
    "CustomerID",
    "TransactionType",
    "Amount",
    "PaymentMethod",
    "TransactionStatus"
).show(20, truncate=False)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 48, Finished, Available, Finished, False)

+-------------+---------+----------+---------------+------+-------------+-----------------+
|TransactionID|AccountID|CustomerID|TransactionType|Amount|PaymentMethod|TransactionStatus|
+-------------+---------+----------+---------------+------+-------------+-----------------+
|TXN001352    |ACC07492 |CUST01263 |Payment        |NULL  |Cash         |Completed        |
|TXN005765    |ACC13847 |CUST09176 |Withdrawal     |NULL  |UPI          |Completed        |
|TXN016615    |ACC01986 |CUST04124 |Deposit        |NULL  |Credit Card  |Completed        |
|TXN024830    |ACC05499 |CUST00479 |Payment        |NULL  |Debit Card   |Completed        |
|TXN029380    |ACC05960 |CUST09877 |Withdrawal     |NULL  |IMPS         |Completed        |
|TXN031819    |ACC10580 |CUST09353 |Transfer       |NULL  |UPI          |Completed        |
|TXN037601    |ACC14188 |CUST03496 |Withdrawal     |NULL  |UPI          |Completed        |
|TXN040843    |ACC14175 |CUST07044 |Withdrawal     |NULL  |UPI          |Complet

In [47]:
df_silver = df_silver.filter(
    col("Amount").isNotNull()
)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 49, Finished, Available, Finished, False)

In [48]:
df_silver.filter(
    col("Amount").isNull()
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 50, Finished, Available, Finished, False)

0

In [49]:
df_silver.filter(
    col("PaymentMethod").isNull()
).select(
    "TransactionID",
    "AccountID",
    "CustomerID",
    "TransactionType",
    "Amount",
    "TransactionStatus"
).show(20, truncate=False)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 51, Finished, Available, Finished, False)

+-------------+---------+----------+---------------+--------+-----------------+
|TransactionID|AccountID|CustomerID|TransactionType|Amount  |TransactionStatus|
+-------------+---------+----------+---------------+--------+-----------------+
|TXN001102    |ACC04004 |CUST08345 |Transfer       |830.31  |Completed        |
|TXN002913    |ACC08826 |CUST00175 |Withdrawal     |1474.4  |Completed        |
|TXN005298    |ACC05767 |CUST02122 |Transfer       |1142.79 |Completed        |
|TXN015136    |ACC01941 |CUST03808 |Payment        |3627.34 |Completed        |
|TXN019875    |ACC01616 |CUST09912 |Deposit        |1284.32 |Completed        |
|TXN020396    |ACC09212 |CUST05109 |Withdrawal     |4540.97 |Completed        |
|TXN021001    |ACC12968 |CUST00319 |Transfer       |2273.24 |Completed        |
|TXN021880    |ACC05838 |CUST08397 |Deposit        |4741.57 |Completed        |
|TXN022344    |ACC04528 |CUST04035 |Transfer       |1038.45 |Failed           |
|TXN022555    |ACC04450 |CUST06968 |Paym

In [50]:
from pyspark.sql.functions import col, when

df_silver = df_silver.withColumn(
    "PaymentMethod",
    when(
        col("PaymentMethod").isNull(),
        "Unknown"
    ).otherwise(col("PaymentMethod"))
)

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 52, Finished, Available, Finished, False)

In [51]:
df_silver.filter(
    col("PaymentMethod").isNull()
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 53, Finished, Available, Finished, False)

0

In [52]:
df_silver.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_silver.columns
]).show()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 54, Finished, Available, Finished, False)

+-------------+---------+----------+---------------+---------------+------+-------------+--------+--------+-----------------+
|TransactionID|AccountID|CustomerID|TransactionDate|TransactionType|Amount|PaymentMethod|BranchID|Location|TransactionStatus|
+-------------+---------+----------+---------------+---------------+------+-------------+--------+--------+-----------------+
|            0|        0|         0|              0|              0|     0|            0|       0|       0|                0|
+-------------+---------+----------+---------------+---------------+------+-------------+--------+--------+-----------------+



In [53]:
df_silver.groupBy("TransactionID") \
    .count() \
    .filter(col("count") > 1) \
    .count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 55, Finished, Available, Finished, False)

0

In [54]:
df_silver.filter(
    col("Amount") < 0
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 56, Finished, Available, Finished, False)

0

In [55]:
df_silver.filter(
    col("Amount") == 0
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 57, Finished, Available, Finished, False)

0

In [56]:
df_silver.filter(
    ~col("TransactionType").isin(
        "Deposit",
        "Withdrawal",
        "Transfer",
        "Payment"
    )
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 58, Finished, Available, Finished, False)

0

In [57]:
df_silver.filter(
    ~col("TransactionStatus").isin(
        "Completed",
        "Failed",
        "Pending"
    )
).count()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 59, Finished, Available, Finished, False)

0

In [58]:
print("Final Silver row count:", df_silver.count())

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 60, Finished, Available, Finished, False)

Final Silver row count: 99900


In [59]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.transactions")

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 61, Finished, Available, Finished, False)

In [60]:
df_silver_check = spark.read.table("silver.transactions")

print("Saved Silver row count:", df_silver_check.count())

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 62, Finished, Available, Finished, False)

Saved Silver row count: 99900


In [61]:
df_gold = spark.read.table("silver.transactions")

df_gold.printSchema()

StatementMeta(, 51f53671-cb13-49c4-a4ea-bac57cf6ddbb, 63, Finished, Available, Finished, False)

root
 |-- TransactionID: string (nullable = true)
 |-- AccountID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- TransactionType: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- BranchID: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- TransactionStatus: string (nullable = true)



In [1]:
df_gold = spark.read.table("silver.transactions")

df_gold.printSchema()

StatementMeta(, d3b48e69-4cd3-422e-983b-7b4142b93ac7, 3, Finished, Available, Finished, False)

root
 |-- TransactionID: string (nullable = true)
 |-- AccountID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- TransactionType: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- BranchID: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- TransactionStatus: string (nullable = true)



In [2]:
from pyspark.sql.functions import count, sum

df_gold_transaction_summary = df_gold.groupBy(
    "TransactionType"
).agg(
    count("TransactionID").alias("TotalTransactions"),
    sum("Amount").alias("TotalAmount")
).orderBy("TransactionType")

df_gold_transaction_summary.show()

StatementMeta(, d3b48e69-4cd3-422e-983b-7b4142b93ac7, 5, Finished, Available, Finished, False)

+---------------+-----------------+--------------------+
|TransactionType|TotalTransactions|         TotalAmount|
+---------------+-----------------+--------------------+
|        Deposit|            27978|1.5291776624999988E8|
|        Payment|            19916|1.0715426307999998E8|
|       Transfer|            27080|1.5038065792999995E8|
|     Withdrawal|            24926|1.3416346430000006E8|
+---------------+-----------------+--------------------+



In [3]:
df_gold_payment_summary = df_gold.groupBy(
    "PaymentMethod"
).agg(
    count("TransactionID").alias("TotalTransactions"),
    sum("Amount").alias("TotalAmount")
).orderBy("PaymentMethod")

df_gold_payment_summary.show()

StatementMeta(, d3b48e69-4cd3-422e-983b-7b4142b93ac7, 8, Finished, Available, Finished, False)

+-------------+-----------------+--------------------+
|PaymentMethod|TotalTransactions|         TotalAmount|
+-------------+-----------------+--------------------+
|          ATM|            11841|       6.344484823E7|
|         Cash|             8044|4.4467993730000004E7|
|  Credit Card|             8043| 4.289143448000001E7|
|   Debit Card|            12134|       6.717390308E7|
|         IMPS|            12853| 7.076635596000002E7|
|         NEFT|            11999| 6.556367483000004E7|
|          UPI|            34738|1.8900620857000008E8|
|      Unknown|              248|          1301732.68|
+-------------+-----------------+--------------------+



In [4]:
df_gold_location_summary = df_gold.groupBy(
    "Location"
).agg(
    count("TransactionID").alias("TotalTransactions"),
    sum("Amount").alias("TotalAmount")
).orderBy("Location")

df_gold_location_summary.show()

StatementMeta(, d3b48e69-4cd3-422e-983b-7b4142b93ac7, 11, Finished, Available, Finished, False)

+----------+-----------------+--------------------+
|  Location|TotalTransactions|         TotalAmount|
+----------+-----------------+--------------------+
|     Delhi|            12045| 6.700875023000007E7|
| Hyderabad|            13447| 7.441093935000007E7|
| Mangaluru|            12575| 6.819239848999995E7|
|      Pune|            12875| 6.849341351999994E7|
|     Salem|            12517| 6.760523802000004E7|
|  Siliguri|            11927|6.5389509899999976E7|
|  Vadodara|            12208| 6.595511387000002E7|
|Vijayawada|            12306| 6.756078818000005E7|
+----------+-----------------+--------------------+



In [5]:
df_gold_status_summary = df_gold.groupBy(
    "TransactionStatus"
).agg(
    count("TransactionID").alias("TotalTransactions"),
    sum("Amount").alias("TotalAmount")
).orderBy("TransactionStatus")

df_gold_status_summary.show()

StatementMeta(, d3b48e69-4cd3-422e-983b-7b4142b93ac7, 14, Finished, Available, Finished, False)

+-----------------+-----------------+--------------------+
|TransactionStatus|TotalTransactions|         TotalAmount|
+-----------------+-----------------+--------------------+
|        Completed|            93802|5.1162161819000006E8|
|           Failed|             3095|1.6726900649999999E7|
|          Pending|             3003|       1.626763272E7|
+-----------------+-----------------+--------------------+



In [6]:
df_gold_transaction_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.transaction_summary")

df_gold_payment_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.payment_summary")

df_gold_location_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.location_summary")

df_gold_status_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.status_summary")

StatementMeta(, d3b48e69-4cd3-422e-983b-7b4142b93ac7, 16, Finished, Available, Finished, False)

In [7]:
spark.sql("SHOW TABLES IN gold").show(truncate=False)

StatementMeta(, d3b48e69-4cd3-422e-983b-7b4142b93ac7, 17, Finished, Available, Finished, False)

+--------------------------------------------------+-------------------+-----------+
|namespace                                         |tableName          |isTemporary|
+--------------------------------------------------+-------------------+-----------+
|Banking_Financial_Analytics.Banking_Lakehouse.gold|account_summary    |false      |
|Banking_Financial_Analytics.Banking_Lakehouse.gold|branch_summary     |false      |
|Banking_Financial_Analytics.Banking_Lakehouse.gold|customer_summary   |false      |
|Banking_Financial_Analytics.Banking_Lakehouse.gold|date_summary       |false      |
|Banking_Financial_Analytics.Banking_Lakehouse.gold|location_summary   |false      |
|Banking_Financial_Analytics.Banking_Lakehouse.gold|payment_summary    |false      |
|Banking_Financial_Analytics.Banking_Lakehouse.gold|status_summary     |false      |
|Banking_Financial_Analytics.Banking_Lakehouse.gold|transaction_summary|false      |
+--------------------------------------------------+-------------

In [8]:
spark.sql("""
SELECT
    (SELECT COUNT(*) FROM gold.transaction_summary) AS transaction_summary_rows,
    (SELECT COUNT(*) FROM gold.payment_summary) AS payment_summary_rows,
    (SELECT COUNT(*) FROM gold.location_summary) AS location_summary_rows,
    (SELECT COUNT(*) FROM gold.status_summary) AS status_summary_rows
""").show()

StatementMeta(, d3b48e69-4cd3-422e-983b-7b4142b93ac7, 18, Finished, Available, Finished, False)

+------------------------+--------------------+---------------------+-------------------+
|transaction_summary_rows|payment_summary_rows|location_summary_rows|status_summary_rows|
+------------------------+--------------------+---------------------+-------------------+
|                       4|                   8|                    8|                  3|
+------------------------+--------------------+---------------------+-------------------+

